# Lottery Analysis & Prediction Dashboard

In [1]:
import sys
import os
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# Robustly find the Modules directory
possible_paths = [
    os.path.abspath('.'),
    '/mnt/c/Users/ag/Documents/Obsidian/PKM/Workspace/(P) Python Prediction Engine/Modules',
    os.path.abspath('PKM/Workspace/(P) Python Prediction Engine/Modules'),
    os.path.abspath('../Modules')
]

for path in possible_paths:
    if os.path.exists(os.path.join(path, 'main_engine.py')):
        if path not in sys.path:
            sys.path.append(path)
        os.chdir(path) # Ensure data paths work correctly
        break

try:
    from main_engine import PredictionEngine
    from statistics_analyzer import StatisticsAnalyzer
    print(f"[+] Successfully loaded modules from: {os.getcwd()}")
except ImportError as e:
    print(f"[-] Error: Could not find modules. {e}")
    print(f"Current sys.path: {sys.path}")

[+] Successfully loaded modules from: /mnt/c/Users/ag/Documents/Obsidian/PKM/Workspace/(P) Python Prediction Engine/Modules


## Dashboard Interface

In [2]:
class LotteryDashboard:
    def __init__(self):
        self.output = widgets.Output()
        self.game_type = "Lotofácil"
        
        # Check if we are in a widget-capable environment
        print(f"[+] Initializing Dashboard for {self.game_type}...")
        
        self.selector = widgets.Dropdown(
            options=['Lotofácil', 'Mega-Sena'],
            value='Lotofácil', 
            description='Lottery:',
        )
        self.selector.observe(self.on_change, names='value')
        
        self.refresh_btn = widgets.Button(description="Run Analysis", button_style='primary')
        self.refresh_btn.on_click(self.run_all)
        
        # Explicitly display the UI components
        ui = widgets.HBox([self.selector, self.refresh_btn])
        display(ui)
        display(self.output)
        
        self.run_all()

    def on_change(self, change):
        self.game_type = change['new']
        self.run_all()

    def run_all(self, b=None):
        with self.output:
            clear_output(wait=True)
            self.engine = PredictionEngine(self.game_type)
            data = self.engine.load_data()

            if not data:
                print("[-] No data found.")
                return

            self.display_historical(data)
            self.display_stats(data)
            self.display_prediction(data)

    def display_historical(self, data):
        print(f"### Historical Data: {self.game_type} ###")
        df = pd.DataFrame([{'Date': d['date'], 'Numbers': sorted(d['nums'])} for d in data[-10:]])
        display(df.sort_values('Date', ascending=False))

        # Frequency Heatmap
        all_nums = []
        for d in data: all_nums.extend(d['nums'])
        counts = pd.Series(all_nums).value_counts().sort_index()

        plt.figure(figsize=(12, 4))
        sns.barplot(x=counts.index, y=counts.values, hue=counts.index, palette="viridis", legend=False)
        plt.title(f"Number Frequency - {self.game_type}")
        plt.xlabel("Number")
        plt.ylabel("Frequency")
        plt.show()

    def display_stats(self, data):
        print(f"### Statistics Summary ###")
        stats_analyzer = StatisticsAnalyzer()
        last_draw = data[-1]['nums']
        prev_draw = data[-2]['nums'] if len(data) > 1 else None

        stats_analyzer.print_profile(last_draw, prev_draw, title="Last Draw Profile")

        # Parity Distribution
        even_counts = [len([n for n in d['nums'] if n % 2 == 0]) for d in data[-100:]]
        plt.figure(figsize=(6, 3))
        plt.hist(even_counts, bins=range(min(even_counts), max(even_counts) + 2), alpha=0.7, color='skyblue', edgecolor='black')
        plt.title("Distribution of Even Numbers (Last 100 draws)")
        plt.xlabel("Count of Even Numbers")
        plt.show()

    def display_prediction(self, data):
        print(f"### Prediction Engine Dashboard ###")
        self.engine.run(mode="predict")

dashboard = LotteryDashboard()

Output()